In [5]:
import pyreadstat

path = "../data/raw/CY09_MS_STU_PUF.sav"

df_meta, meta = pyreadstat.read_sav(path, metadataonly=True)

print("Переменных:", len(meta.column_names))
print("Наблюдений:", meta.number_rows)


Переменных: 885
Наблюдений: 755721


In [6]:
ai_vars = {
    name: label
    for name, label in meta.column_names_to_labels.items()
    if label and any(k in label.lower() for k in
                     ["artificial intelligence", "chatbot", "chatgpt", "generative"])
}

print("Найдено:", len(ai_vars))
for name, label in ai_vars.items():
    print(f"{name}: {label}")

Найдено: 9
ST436Q16DA: This school year, how often did you do the following tasks using <digital resources> in your school lessons? - Assess the quality of information generated by Artificial Intelligence chatbots (e.g. <ChatGPT>, <Google Bard>)
ST438Q01DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To summarize a text I had to read
ST438Q02DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To conduct preliminary research on a new topic
ST438Q03DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To draft texts for writing assignments
ST438Q04DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To help me learn
IC170Q10DA: This school year, how often did you use the following

In [9]:
df_ai, _ = pyreadstat.read_sav(
    path, usecols=["CNT", "ST438Q03DA", "ST438Q04DA", "IC170Q10DA", "ESCS"]
)
print(df_ai.notna().mean().round(3))

CNT           1.000
ST438Q03DA    0.821
ST438Q04DA    0.818
IC170Q10DA    0.373
ESCS          0.901
dtype: float64


In [10]:
cov = (df_ai.groupby("CNT")[["ST438Q03DA", "IC170Q10DA", "ESCS"]]
             .apply(lambda g: g.notna().mean())
             .round(2))
cov["n"] = df_ai.groupby("CNT").size()
print(cov.sort_values("n", ascending=False).to_string())

     ST438Q03DA  IC170Q10DA  ESCS      n
CNT                                     
ARE        0.90        0.81  0.91  34639
BRA        0.80        0.00  0.91  30140
ESP        0.88        0.79  0.95  29966
CAN        0.77        0.00  0.90  24601
KAZ        0.92        0.00  0.93  22680
GBR        0.87        0.17  0.90  15546
AUS        0.93        0.89  0.95  14658
IDN        0.96        0.00  0.98  14168
QCI        1.00        0.97  1.00  12385
ARG        0.69        0.51  0.88  11092
PSE        0.78        0.00  0.85   9647
ECU        0.94        0.00  0.97   9128
ITA        0.95        0.87  0.97   8946
UZB        0.92        0.00  0.90   8652
THA        0.97        0.00  0.97   8547
CZE        0.90        0.00  0.95   8462
JOR        0.86        0.57  0.90   8290
PHL        0.97        0.00  0.97   8072
MEX        0.95        0.00  0.98   7843
ROU        0.90        0.76  0.95   7725
LTU        0.89        0.00  0.89   7718
TUR        0.97        0.90  0.97   7702
FIN        0.90 

In [11]:
decision = """# Country selection — PISA 2025

Final sample: ARE, ESP, AUS, BRA, KAZ, GBR

| Country | n | ST438 coverage | IC170 coverage | ESCS coverage |
|---|---|---|---|---|
| ARE | 34,639 | 0.90 | 0.81 | 0.91 |
| BRA | 30,140 | 0.80 | 0.00 | 0.91 |
| ESP | 29,966 | 0.88 | 0.79 | 0.95 |
| KAZ | 22,680 | 0.92 | 0.00 | 0.93 |
| GBR | 15,546 | 0.87 | 0.17 | 0.90 |
| AUS | 14,658 | 0.93 | 0.89 | 0.95 |

Rationale: six world regions (Gulf, Southern Europe, Oceania,
Latin America, Central Asia, North-Western Europe); all six have
ST438 coverage above 0.80 and ESCS above 0.90.

Main specification uses ST438 (how AI is used) and ESCS.
IC170 (school vs home use) is available only in ARE, ESP and AUS,
so it is used for robustness on a sub-sample, never in the main model.

Excluded: USA (ST438 0.51, ESCS 0.55 — coverage too low to include).
Canada dropped in favour of Australia (ST438 0.77 vs 0.93, no ICT block).
"""

with open("../docs/country_selection.md", "w") as f:
    f.write(decision)

print("Сохранено")

Сохранено


In [12]:
pv = [n for n in meta.column_names if n.startswith("PV")]
print("Всего PV-переменных:", len(pv))

domains = sorted({n[3:] for n in pv})
print("Домены:", domains)

Всего PV-переменных: 111
Домены: ['0CMOD', '0CMPS', '0CPPK', '0CPRO', '0MATH', '0READ', '0SCIE', '0SEDE', '0SEID', '0SENV', '0SEPS', 'CMOD', 'CMPS', 'CPPK', 'CPRO', 'MATH', 'READ', 'RSCUSE', 'SCIE', 'SEDE', 'SEID', 'SENV', 'SEPS']


In [13]:
df_pv, _ = pyreadstat.read_sav(
    path, usecols=["CNT", "PV1MATH", "PV1READ", "PV1SCIE", "ST438Q03DA", "ESCS"]
)

six = ["ARE", "ESP", "AUS", "BRA", "KAZ", "GBR"]
sub = df_pv[df_pv["CNT"].isin(six)]

print(sub.groupby("CNT")[["PV1MATH", "PV1READ", "PV1SCIE"]]
         .apply(lambda g: g.notna().mean()).round(3).to_string())
print("\nРазмер подвыборки:", len(sub))

     PV1MATH  PV1READ  PV1SCIE
CNT                           
ARE      1.0      1.0      1.0
AUS      1.0      1.0      1.0
BRA      1.0      1.0      1.0
ESP      1.0      1.0      1.0
GBR      1.0      1.0      1.0
KAZ      1.0      1.0      1.0

Размер подвыборки: 147629


In [14]:
for d in ["RSCUSE", "CMOD", "CPRO", "SEDE"]:
    name = f"PV1{d}"
    print(f"{name}: {meta.column_names_to_labels.get(name)}")
    

PV1RSCUSE: None
PV1CMOD: Plausible Value 1 in LDW Subscale - Modelling 
PV1CPRO: Plausible Value 1 in LDW Subscale - Programming 
PV1SEDE: Plausible Value 1 in Competency Subscale of Science - Evaluate Designs for Scientific Enquiry 


In [15]:
df_ldw, _ = pyreadstat.read_sav(
    path, usecols=["CNT", "PV1CMOD", "PV1CPRO"]
)
ldw = df_ldw[df_ldw["CNT"].isin(six)]
print(ldw.groupby("CNT")[["PV1CMOD", "PV1CPRO"]]
         .apply(lambda g: g.notna().mean()).round(3).to_string())

     PV1CMOD  PV1CPRO
CNT                  
ARE      1.0      1.0
AUS      1.0      1.0
BRA      1.0      1.0
ESP      1.0      1.0
GBR      1.0      1.0
KAZ      1.0      1.0


In [16]:
note = """# Dependent variable — options (PISA 2025)

All six countries have full coverage (1.00) for:
- PV1-10 MATH, READ, SCIE (core domains)
- PV1-10 CMOD (LDW subscale: Modelling)
- PV1-10 CPRO (LDW subscale: Programming)

Sub-sample of the six countries: 147,629 students.

LDW = Learning in the Digital World, new domain in 2025.
Measures capacity to learn with digital tools — conceptually
much closer to the research question than attainment in maths.

Open: what exactly LDW subscales measure — need the 2025
assessment framework before choosing. If LDW fits, it becomes
the main outcome, with MATH as a robustness check.
If not: MATH main (less sensitive to language-of-test issues
in ARE and KAZ), READ as robustness.

ST438 (how AI is used) is asked once about schoolwork in general,
not per subject — there is no domain-specific AI question.
"""

with open("../docs/outcome_variable.md", "w") as f:
    f.write(note)

print("Сохранено")

Сохранено
